# AI Remix Studio — bộ xử lý miễn phí bằng Google Colab

1. Mở notebook này bằng Google Colab.
2. Chọn **Runtime → Change runtime type → T4 GPU** nếu tùy chọn này xuất hiện (GPU miễn phí không được đảm bảo).
3. Chạy lần lượt các ô từ trên xuống dưới.
4. Ở ô cuối, copy URL `https://....trycloudflare.com` và dán vào ô **URL Colab** trên website AI Remix Studio rồi bấm **Kết nối**.

**Lưu ý:** Colab miễn phí có giới hạn phiên, có thể ngắt bất kỳ lúc nào. Khi Colab ngắt, website vẫn mở được nhưng xử lý nhạc sẽ tạm ngưng. File cache và bản render trong phiên Colab có thể mất khi runtime bị reset. Không tải nhạc nhạy cảm lên phiên dùng chung.

In [ ]:
import os, subprocess, pathlib
REPO = "/content/DuyKhanhMusic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Khanhcute111/DuyKhanhMusic.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
print("Đã tải mã nguồn:", REPO)

In [ ]:
import sys, subprocess, pathlib
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
# Giữ PyTorch/Torchaudio GPU có sẵn của Colab, chỉ cài các thư viện còn lại.
requirements = pathlib.Path("engine/requirements.txt").read_text().splitlines()
filtered = [line for line in requirements if not line.strip().lower().startswith(("torch", "torchaudio"))]
pathlib.Path("/content/airemix-colab-requirements.txt").write_text("\n".join(filtered) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/airemix-colab-requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__, " | CUDA:", torch.cuda.is_available())
print("Cài đặt hoàn tất. Lần đầu Demucs có thể tải model htdemucs.")

In [ ]:
import os, re, time, queue, threading, subprocess, urllib.request, pathlib
# Khai báo lại đường dẫn để ô này chạy độc lập, kể cả khi Colab chưa chạy ô đầu.
import os, subprocess, pathlib, sys, re, time, queue, threading, urllib.request
REPO = "/content/DuyKhanhMusic"
if not pathlib.Path(REPO).is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Khanhcute111/DuyKhanhMusic.git", REPO], check=True)
os.chdir(REPO)
# Dừng server/tunnel cũ nếu đang chạy, cập nhật mã nguồn mới nhất rồi khởi động lại.
for proc_name in ("tunnel", "server"):
    proc = globals().get(proc_name)
    try:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=8)
    except Exception:
        try: proc.kill()
        except Exception: pass
subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
PORT = 8000
env = os.environ.copy()
env["AIREMIX_HOST"] = "0.0.0.0"
env["AIREMIX_PORT"] = str(PORT)
server = subprocess.Popen(["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(PORT)], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
# Chờ FastAPI sẵn sàng.
ready = False
for _ in range(90):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/system", timeout=2) as response:
            ready = response.status == 200
        if ready: break
    except Exception:
        time.sleep(2)
if not ready:
    server.terminate()
    raise RuntimeError("FastAPI không khởi động được. Xem lỗi ở ô cài đặt phía trên.")
# Tạo HTTPS tunnel miễn phí tạm thời bằng Cloudflare Quick Tunnel.
cloudflared_path = "/content/cloudflared"
if not pathlib.Path(cloudflared_path).exists():
    subprocess.run(["wget", "-q", "-O", cloudflared_path, "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(cloudflared_path, 0o755)
tunnel = subprocess.Popen([cloudflared_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log_queue = queue.Queue()
def read_tunnel_logs():
    for line in tunnel.stdout:
        log_queue.put(line)
threading.Thread(target=read_tunnel_logs, daemon=True).start()
public_url = None
deadline = time.time() + 90
while time.time() < deadline and public_url is None:
    try:
        line = log_queue.get(timeout=2)
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match: public_url = match.group(0)
    except queue.Empty:
        if tunnel.poll() is not None: break
if not public_url:
    raise RuntimeError("Không lấy được URL tunnel. Hãy chạy lại ô này hoặc thử lại sau.")
print("\nAI REMIX API ĐANG CHẠY:")
print(public_url)
print("\nDán URL trên vào ô URL Colab ở đầu website và bấm Kết nối.")
print("Giữ phiên Colab này mở trong lúc xử lý nhạc.")